# SQL for Data Analysis with SQLite

Use SQL to query tables and combine SQL with pandas without requiring a server.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Create an in-memory SQLite database and inspect tables with pandas.
- Use `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`, and `JOIN`.
- Understand parameterized queries and introductory window functions.

SQLite is a lightweight relational database included with Python. The examples use `:memory:` so they create no database file.

In [1]:
import sqlite3
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer_id": [101, 102, 101, 103, 102],
    "region": ["North", "South", "North", "East", "South"],
    "amount": [120.0, 75.0, 45.0, 200.0, 30.0],
})
customers = pd.DataFrame({
    "customer_id": [101, 102, 103],
    "segment": ["Retail", "Business", "Retail"],
})
conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
display(pd.read_sql_query("SELECT * FROM orders LIMIT 3", conn))

,order_id,customer_id,region,amount
0,1,101,North,120.0
1,2,102,South,75.0
2,3,101,North,45.0


## Filtering and grouping

In [2]:
query = """
SELECT region, COUNT(*) AS order_count, SUM(amount) AS revenue,
       AVG(amount) AS average_order
FROM orders
WHERE amount >= 50
GROUP BY region
ORDER BY revenue DESC
"""
display(pd.read_sql_query(query, conn))

,region,order_count,revenue,average_order
0,East,1,200.0,200.0
1,North,1,120.0,120.0
2,South,1,75.0,75.0


## JOIN and safe parameters
A join may multiply rows when a join key appears more than once on either side. Check key uniqueness when a one-to-one or many-to-one relationship is expected. Use parameters for values rather than string formatting.

In [3]:
join_query = """
SELECT o.order_id, o.region, o.amount, c.segment
FROM orders AS o
LEFT JOIN customers AS c USING (customer_id)
ORDER BY o.order_id
"""
display(pd.read_sql_query(join_query, conn))

minimum_amount = 70
safe_query = "SELECT * FROM orders WHERE amount >= ? ORDER BY amount DESC"
display(pd.read_sql_query(safe_query, conn, params=(minimum_amount,)))

,order_id,region,amount,segment
0,1,North,120.0,Retail
1,2,South,75.0,Business
2,3,North,45.0,Retail
3,4,East,200.0,Retail
4,5,South,30.0,Business


,order_id,customer_id,region,amount
0,4,103,East,200.0
1,1,101,North,120.0
2,2,102,South,75.0


## Window functions
Unlike `GROUP BY`, a window function keeps individual rows while adding a calculation across related rows. SQLite supports common window functions in current versions.

In [4]:
window_query = """
SELECT order_id, region, amount,
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rank_in_region,
       SUM(amount) OVER (PARTITION BY region) AS region_total
FROM orders
ORDER BY region, rank_in_region
"""
display(pd.read_sql_query(window_query, conn))
conn.close()

,order_id,region,amount,rank_in_region,region_total
0,4,East,200.0,1,200.0
1,1,North,120.0,1,165.0
2,3,North,45.0,2,165.0
3,2,South,75.0,1,105.0
4,5,South,30.0,2,105.0


## Practice
Write a SQL query to return revenue by customer and order it from highest to lowest. Think about whether the result should contain one row per customer or one row per order.

In [5]:
conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False, if_exists="replace")
query = "SELECT customer_id, SUM(amount) AS total_spend FROM orders GROUP BY customer_id ORDER BY total_spend DESC"
display(pd.read_sql_query(query, conn))
conn.close()

,customer_id,total_spend
0,103,200.0
1,101,165.0
2,102,105.0
